# Практическое занятие 10. Создание объектов Spark DataFrame и выполнение основных операций в Apache Spark

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkb40-svg/big_data_6semestr/blob/main/notebooks/p10.ipynb)

**Цели занятия:**

- Создавать Spark DataFrame из списков, словарей, объектов `Row`, pandas и с явной схемой `StructType`
- Выбирать и фильтровать данные: `select`, `alias`, `filter`/`where` с условиями `&`, `|`, `~`
- Добавлять, переименовывать и удалять столбцы: `withColumn`, `when`/`otherwise`, `lit`, `withColumnRenamed`, `drop`
- Группировать, сортировать и соединять таблицы, получать результаты в Python: `groupBy().agg`, `join`, `collect`, `first`

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/mkb40-svg/big_data_6semestr.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"

# Для заданий по Spark нужен настоящий PySpark (в Colab ставится за ~1 минуту)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Способы создать DataFrame

Spark DataFrame — распределённая таблица со **схемой**: у каждого столбца есть имя и тип. Создать её можно из обычных объектов Python:

| Источник | Пример | Имена столбцов |
|---|---|---|
| список кортежей + имена | `spark.createDataFrame([("Aziza", 3)], ["name", "course"])` | из списка имён |
| список словарей | `spark.createDataFrame([{"city": "Toshkent", "population": 3040000}])` | ключи словаря |
| объекты `Row` | `spark.createDataFrame([Row(title="O'tkan kunlar", year=1926)])` | поля Row |
| схема `StructType` | `spark.createDataFrame(rows, schema)` | из схемы, типы строго заданы |
| DDL-строка | `spark.createDataFrame(rows, "id INT, name STRING")` | из строки |
| pandas DataFrame | `spark.createDataFrame(pdf)` | столбцы pandas |
| файл | `spark.read.csv(...)`, `spark.read.json(...)` | заголовок / ключи JSON |

Если схема не задана, Spark **выводит типы** по значениям: `int` Python → `bigint` (LongType), `float` → `double`, `str` → `string`, `datetime.date` → `date`.

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("id", IntegerType(), nullable=False),   # NOT NULL
    StructField("name", StringType(), True),
    StructField("gpa", DoubleType(), True),
])
df = spark.createDataFrame([(1, "Aziza Karimova", 4.62)], schema)
df.printSchema()
```

## Обращение к столбцам

Четыре равноценные записи: `"gpa"`, `F.col("gpa")`, `df.gpa`, `df["gpa"]`. Строку можно передавать в `select`, `groupBy`, `orderBy`; для выражений (арифметика, сравнения) нужен объект **Column** — `F.col("gpa") * 25`.

## Основные операции

| Операция | Пример |
|---|---|
| выбор и переименование | `df.select("name", F.col("gpa").alias("ball"))` |
| фильтр | `df.filter(F.col("gpa") >= 4)` или `df.where("gpa >= 4")` |
| новый столбец | `df.withColumn("gpa100", F.col("gpa") / 5 * 100)` |
| константа | `df.withColumn("year", F.lit("2025/2026"))` |
| условие CASE WHEN | `F.when(F.col("gpa") >= 4.5, "a'lo").otherwise("yaxshi")` |
| переименование | `df.withColumnRenamed("name", "full_name")` |
| удаление столбцов | `df.drop("city")` |
| уникальные строки | `df.select("faculty").distinct()` |
| сортировка | `df.orderBy(F.desc("gpa"), "name")` |
| первые N строк | `df.limit(3)` |
| группировка | `df.groupBy("faculty").agg(F.count("*").alias("n"), F.avg("gpa"))` |
| соединение | `students.join(faculties, on="faculty_id", how="left")` |

> **Внимание: Логические операции над столбцами**  
> В условиях используют `&` (и), `|` (или), `~` (не), а **каждое сравнение берут в скобки**:
> `df.filter((F.col("gpa") >= 4) & (F.col("course") == 3))`.
> Слова `and`, `or`, `not` со столбцами не работают — Python попытается превратить Column в True/False и выдаст ошибку.

## Неизменяемость

DataFrame **неизменяем**: каждая трансформация возвращает **новый** DataFrame, а исходный остаётся прежним. Поэтому результат нужно присвоить переменной, а операции удобно записывать цепочкой:

```python
report = (students
          .withColumn("gpa100", F.round(F.col("gpa") / 5 * 100, 1))
          .withColumnRenamed("name", "full_name")
          .drop("city")
          .orderBy(F.desc("gpa")))
```

## Соединения (join)

| how | Что остаётся |
|---|---|
| `inner` (по умолчанию) | только пары с совпадающим ключом |
| `left` | все строки левой таблицы; нет пары → `null` |
| `right` | все строки правой таблицы |
| `outer` / `full` | все строки обеих таблиц |
| `left_anti` | строки левой таблицы, у которых **нет** пары |

## Из Spark в Python

Действия возвращают данные в драйвер:

- `df.collect()` — **список** объектов `Row` (осторожно с большими таблицами — всё попадёт в память драйвера!);
- `df.first()` / `df.head()` — одна строка `Row`; `df.take(5)` — список из 5 строк;
- `df.count()` — число строк;
- `df.toPandas()` — pandas DataFrame (тоже только для небольших результатов).

Поле `Row` читается по имени (`row["name"]` или `row.name`) или по номеру (`row[0]`); `row.asDict()` превращает строку в словарь.

```python
best = df.orderBy(F.desc("gpa")).first()
print(best["name"], best.gpa)
names = [r["name"] for r in df.select("name").collect()]
```

> **Заметка: Тренажёр**  
> Задания выполняются в учебном эмуляторе PySpark (тот же API, что в PySpark 3.5). Обратите внимание: в настоящем Spark DataFrame из **словарей** получает столбцы в алфавитном порядке ключей.

## Задание 1. DataFrame из списка кортежей

Создайте DataFrame `students` из списка кортежей `data` и списка имён столбцов: `name`, `faculty`, `course`, `gpa`.

Выведите таблицу (`show()`) и схему (`printSchema()`), сохраните число строк в `n`. Посмотрите, какие типы Spark выбрал для `course` и `gpa`.

<details><summary>💡 Подсказка</summary>

`columns = ["name", "faculty", "course", "gpa"]`, `students = spark.createDataFrame(data, columns)`, `n = students.count()`.

</details>

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("UzBigData-P10").getOrCreate()

data = [
    ("Aziza Karimova", "Kompyuter lingvistikasi", 3, 4.62),
    ("Jasur Toshmatov", "Tarjimonlik", 2, 3.85),
    ("Malika Rahimova", "Filologiya", 3, 4.10),
    ("Otabek Saidov", "Kompyuter lingvistikasi", 1, 3.40),
    ("Dilnoza Yusupova", "Xalqaro jurnalistika", 4, 4.95),
    ("Sardor Aliyev", "Tarjimonlik", 3, 2.90),
]
columns = ...     # TODO: список имён столбцов

students = ...    # TODO: spark.createDataFrame(...)
students.show()
students.printSchema()

n = ...
print("Студентов:", n)

## Задание 2. Из словарей, Row и pandas

Создайте три DataFrame разными способами:

1. `df_dict` — из списка словарей `cities` (города Узбекистана).
2. `df_row` — из списка объектов `Row` `books` (узбекская классика). `Book = Row("title", "author", "year")` — это «шаблон» строки с именованными полями.
3. `df_pd` — из pandas DataFrame `pdf` (частоты слов).

Для каждого выведите `show()` и список типов `dtypes`.

<details><summary>💡 Подсказка</summary>

Во всех трёх случаях подходит `spark.createDataFrame(...)`: `spark.createDataFrame(cities)`, `spark.createDataFrame(books)`, `spark.createDataFrame(pdf)`.

</details>

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession, Row

spark = SparkSession.builder.getOrCreate()

cities = [
    {"city": "Toshkent", "region": "Toshkent shahri", "population": 3040000},
    {"city": "Samarqand", "region": "Samarqand", "population": 573000},
    {"city": "Buxoro", "region": "Buxoro", "population": 285000},
    {"city": "Nukus", "region": "Qoraqalpog'iston", "population": 330000},
]
df_dict = ...

Book = Row("title", "author", "year")
books = [
    Book("O'tkan kunlar", "Abdulla Qodiriy", 1926),
    Book("Mehrobdan chayon", "Abdulla Qodiriy", 1929),
    Book("Kecha va kunduz", "Cho'lpon", 1936),
]
df_row = ...

pdf = pd.DataFrame({"word": ["kitob", "maktab", "talaba", "ilm"], "freq": [42, 35, 28, 19]})
df_pd = ...

for df in [df_dict, df_row, df_pd]:
    df.show()
    print(df.dtypes)

## Задание 3. Явная схема: StructType и DDL

Когда типы важны (например, `id` должен быть 32-битным `int` без пропусков, а дата — типом `date`), схему задают явно.

1. Опишите `schema` через `StructType`: `id` — `IntegerType`, **not null** (`nullable=False`); `name` — `StringType`; `birth_date` — `DateType`; `gpa` — `DoubleType`.
2. Создайте `df` из списка `rows` с этой схемой и выведите `printSchema()`.
3. Создайте такой же `df_ddl`, задав схему DDL-строкой `"id INT, name STRING, birth_date DATE, gpa DOUBLE"`.

<details><summary>💡 Подсказка</summary>

`StructField("id", IntegerType(), False)`, `StructField("birth_date", DateType(), True)`; `df = spark.createDataFrame(rows, schema)`, `df_ddl = spark.createDataFrame(rows, "id INT, name STRING, birth_date DATE, gpa DOUBLE")`.

</details>

In [ ]:
import datetime as dt
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType

spark = SparkSession.builder.getOrCreate()

rows = [
    (1, "Aziza Karimova", dt.date(2005, 3, 14), 4.62),
    (2, "Jasur Toshmatov", dt.date(2006, 7, 2), 3.85),
    (3, "Malika Rahimova", dt.date(2005, 11, 23), 4.10),
    (4, "Otabek Saidov", dt.date(2007, 1, 9), 3.40),
    (5, "Dilnoza Yusupova", dt.date(2004, 5, 30), 4.95),
]

schema = StructType([
    # TODO: 4 поля StructField(имя, тип, nullable)
])
df = ...
# df.printSchema()

df_ddl = ...

## Задание 4. select и filter: выбор строк и столбцов

DataFrame `students` (10 студентов: `id, name, faculty, course, gpa, city`) уже создан в подготовительном коде.

1. `names` — только два столбца: `name` под новым именем **`talaba`** (`alias`) и `faculty`.
2. `excellent` — отличники: `gpa` не ниже 4.5 (`filter`).
3. `kl_senior` — студенты факультета `Kompyuter lingvistikasi` **и** со 2-го курса и старше (`where`, оператор `&`).
4. `tash_sam` — студенты из Ташкента **или** Самарканда (оператор `|`, города: `Toshkent`, `Samarqand`).

Выведите каждый результат методом `show()`.

<details><summary>💡 Подсказка</summary>

`students.select(F.col("name").alias("talaba"), "faculty")`; `students.where((F.col("faculty") == "Kompyuter lingvistikasi") & (F.col("course") >= 2))` — каждое сравнение в скобках.

</details>

In [ ]:
# Подготовительный код задания
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
students = spark.createDataFrame([
    (1, "Aziza Karimova", "Kompyuter lingvistikasi", 3, 4.62, "Toshkent"),
    (2, "Jasur Toshmatov", "Tarjimonlik", 2, 3.85, "Samarqand"),
    (3, "Malika Rahimova", "Filologiya", 3, 4.10, "Buxoro"),
    (4, "Otabek Saidov", "Kompyuter lingvistikasi", 1, 3.40, "Andijon"),
    (5, "Dilnoza Yusupova", "Xalqaro jurnalistika", 4, 4.95, "Toshkent"),
    (6, "Sardor Aliyev", "Tarjimonlik", 3, 2.90, "Namangan"),
    (7, "Nodira Ergasheva", "Filologiya", 2, 4.48, "Samarqand"),
    (8, "Bekzod Nazarov", "Kompyuter lingvistikasi", 4, 3.95, "Farg'ona"),
    (9, "Gulnoza Qodirova", "Xalqaro jurnalistika", 1, 4.20, "Toshkent"),
    (10, "Ulug'bek Mirzayev", "Kompyuter lingvistikasi", 2, 4.75, "Xorazm"),
], ["id", "name", "faculty", "course", "gpa", "city"])

In [ ]:
from pyspark.sql import functions as F

names = students.select(...)
excellent = ...
kl_senior = ...
tash_sam = ...

for df in [names, excellent, kl_senior, tash_sam]:
    df.show()

## Задание 5. Новые столбцы: withColumn, when, lit

Подготовьте из `students` ведомость стипендий `report` — одной цепочкой преобразований:

1. `gpa_100` — балл по 100-балльной шкале: `gpa / 5 * 100`, округлённый до 1 знака (`F.round`).
2. `stipend` — стипендия в сумах: при `gpa >= 4.5` — **1 500 000**, при `gpa >= 4.0` — **1 000 000**, иначе **0** (`F.when(...).when(...).otherwise(...)`).
3. `academic_year` — одинаковое для всех значение `"2025/2026"` (`F.lit`).
4. Переименуйте `name` в `full_name` и удалите столбец `city`.

Выведите `report.show()`.

<details><summary>💡 Подсказка</summary>

`.withColumn("stipend", F.when(F.col("gpa") >= 4.5, 1500000).when(F.col("gpa") >= 4.0, 1000000).otherwise(0))`, `.withColumn("academic_year", F.lit("2025/2026"))`, `.withColumnRenamed("name", "full_name").drop("city")`.

</details>

In [ ]:
# Подготовительный код задания
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
students = spark.createDataFrame([
    (1, "Aziza Karimova", "Kompyuter lingvistikasi", 3, 4.62, "Toshkent"),
    (2, "Jasur Toshmatov", "Tarjimonlik", 2, 3.85, "Samarqand"),
    (3, "Malika Rahimova", "Filologiya", 3, 4.10, "Buxoro"),
    (4, "Otabek Saidov", "Kompyuter lingvistikasi", 1, 3.40, "Andijon"),
    (5, "Dilnoza Yusupova", "Xalqaro jurnalistika", 4, 4.95, "Toshkent"),
    (6, "Sardor Aliyev", "Tarjimonlik", 3, 2.90, "Namangan"),
    (7, "Nodira Ergasheva", "Filologiya", 2, 4.48, "Samarqand"),
    (8, "Bekzod Nazarov", "Kompyuter lingvistikasi", 4, 3.95, "Farg'ona"),
    (9, "Gulnoza Qodirova", "Xalqaro jurnalistika", 1, 4.20, "Toshkent"),
    (10, "Ulug'bek Mirzayev", "Kompyuter lingvistikasi", 2, 4.75, "Xorazm"),
], ["id", "name", "faculty", "course", "gpa", "city"])

In [ ]:
from pyspark.sql import functions as F

report = (students
          .withColumn("gpa_100", ...)
          # TODO: stipend, academic_year, переименование, удаление city
         )
report.show()

## Задание 6. distinct, orderBy и groupBy().agg

1. `faculties` — список **различных** факультетов (один столбец `faculty`, `distinct`).
2. `n_cities` — **число** различных городов (целое число Python).
3. `top3` — 3 студента с наибольшим GPA (`orderBy` по убыванию + `limit`).
4. `stats` — по каждому факультету: `n` — число студентов, `avg_gpa` — средний GPA, округлённый до 2 знаков, `max_gpa` — максимальный GPA. Сортировка — по убыванию `avg_gpa`.

Выведите результаты.

<details><summary>💡 Подсказка</summary>

`students.select("faculty").distinct()`; `students.select("city").distinct().count()`; `students.orderBy(F.desc("gpa")).limit(3)`; `.agg(F.count("*").alias("n"), F.round(F.avg("gpa"), 2).alias("avg_gpa"), F.max("gpa").alias("max_gpa")).orderBy(F.desc("avg_gpa"))`.

</details>

In [ ]:
# Подготовительный код задания
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
students = spark.createDataFrame([
    (1, "Aziza Karimova", "Kompyuter lingvistikasi", 3, 4.62, "Toshkent"),
    (2, "Jasur Toshmatov", "Tarjimonlik", 2, 3.85, "Samarqand"),
    (3, "Malika Rahimova", "Filologiya", 3, 4.10, "Buxoro"),
    (4, "Otabek Saidov", "Kompyuter lingvistikasi", 1, 3.40, "Andijon"),
    (5, "Dilnoza Yusupova", "Xalqaro jurnalistika", 4, 4.95, "Toshkent"),
    (6, "Sardor Aliyev", "Tarjimonlik", 3, 2.90, "Namangan"),
    (7, "Nodira Ergasheva", "Filologiya", 2, 4.48, "Samarqand"),
    (8, "Bekzod Nazarov", "Kompyuter lingvistikasi", 4, 3.95, "Farg'ona"),
    (9, "Gulnoza Qodirova", "Xalqaro jurnalistika", 1, 4.20, "Toshkent"),
    (10, "Ulug'bek Mirzayev", "Kompyuter lingvistikasi", 2, 4.75, "Xorazm"),
], ["id", "name", "faculty", "course", "gpa", "city"])

In [ ]:
from pyspark.sql import functions as F

faculties = ...
n_cities = ...
top3 = ...
stats = (students.groupBy("faculty")
         .agg(
             # TODO: F.count(...).alias("n"), ...
         ))

faculties.show()
print("Городов:", n_cities)
top3.show()
stats.show()

## Задание 7. join студентов и факультетов, collect и first

Даны две таблицы (см. подготовительный код): `students` (`id, name, faculty_id, gpa`) и справочник `faculties` (`faculty_id, faculty_name, building`). У студентки `Nodira Ergasheva` код факультета 105, которого нет в справочнике, а на факультете 106 нет студентов.

1. `inner` — внутреннее соединение по `faculty_id`.
2. `left` — левое соединение: остаются **все** студенты.
3. `no_faculty` — **список Python** имён студентов, для которых факультет не найден (в `left` у них `faculty_name` равен `null` → `isNull()`, затем `collect()`).
4. `best` — строка (`Row`) студента с наибольшим GPA из `inner` (`first()`); `best_name` и `best_faculty` — его имя и название факультета.
5. `by_building` — по каждому корпусу (`building`) из `inner`: `students` — число студентов, `avg_gpa` — средний GPA (2 знака); сортировка по `building`.

<details><summary>💡 Подсказка</summary>

`students.join(faculties, on="faculty_id", how="inner")`, для left — `how="left"`; `[r["name"] for r in left.filter(F.col("faculty_name").isNull()).select("name").collect()]`; `best = inner.orderBy(F.desc("gpa")).first()`.

</details>

In [ ]:
# Подготовительный код задания
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
students = spark.createDataFrame([
    (1, "Aziza Karimova", 101, 4.62),
    (2, "Jasur Toshmatov", 102, 3.85),
    (3, "Malika Rahimova", 103, 4.10),
    (4, "Otabek Saidov", 101, 3.40),
    (5, "Dilnoza Yusupova", 104, 4.95),
    (6, "Sardor Aliyev", 102, 2.90),
    (7, "Nodira Ergasheva", 105, 4.48),
], ["id", "name", "faculty_id", "gpa"])
faculties = spark.createDataFrame([
    (101, "Kompyuter lingvistikasi", "A bino"),
    (102, "Tarjimonlik", "B bino"),
    (103, "Filologiya", "B bino"),
    (104, "Xalqaro jurnalistika", "C bino"),
    (106, "Sharqshunoslik", "D bino"),
], ["faculty_id", "faculty_name", "building"])

In [ ]:
from pyspark.sql import functions as F

inner = ...
left = ...

no_faculty = ...     # список строк Python

best = ...
best_name = ...
best_faculty = ...

by_building = ...

inner.show()
left.show()
print("Без факультета:", no_faculty)
print("Лучший студент:", best_name, "-", best_faculty)

## Самостоятельная работа

## Самостоятельная работа: «Изучение возможностей Spark DataFrame»

1. Составьте шпаргалку по 20 функциям модуля `pyspark.sql.functions` (строковые, даты, массивы, условия, агрегаты) — с примером на данных студентов.
2. Изучите оконные функции (`Window.partitionBy(...).orderBy(...)`, `row_number`, `rank`, `lag`) и найдите лучшего студента **на каждом факультете**.
3. Сравните три способа сделать одно и то же — DataFrame API, `spark.sql` и `selectExpr` — на задаче «средний GPA по факультетам» и посмотрите `explain()` для каждого.
4. **Практика:** загрузите `data/spark/students.jsonl` (`spark.read.json`), соедините с таблицей факультетов, придуманной вами, и сохраните отчёт в Parquet с разбиением по факультету: `df.write.partitionBy("faculty").parquet(...)`.